# The Machine Learning CI/CD Tutorial: Everything You NEED to Know

Welcome to the intersection of Software Engineering and Data Science. Continuous Integration and Continuous Deployment (CI/CD) in standard software means automatically testing code and pushing it to production. In Machine Learning (often called MLOps), this is more complex because your system's behavior depends on **code, data, and model weights**. 

A proper ML CI/CD pipeline ensures that:
1. **CI (Continuous Integration):** New data or model code is automatically tested for quality (e.g., "Does the new model hit at least 85% accuracy?" "Are there null values in the incoming data?").
2. **CD (Continuous Deployment):** If the model passes the tests, it is automatically packaged and deployed to an endpoint where applications can use it.

In this tutorial, we will build the core components of an ML pipeline using **scikit-learn** (for modeling), **MLflow** (for experiment tracking and model registry), **pytest** (for CI testing), and **FastAPI** (for CD serving).

*Note: In a true production environment, these cells would be separate `.py` scripts triggered by a tool like GitHub Actions. For learning purposes, we are simulating the pipeline steps here.*

In [ ]:
# Install the necessary libraries for our CI/CD pipeline
# We use ! to run shell commands within a Jupyter Notebook environment
pip install -q scikit-learn mlflow pytest fastapi uvicorn pandas numpy

## Step 1: Data Ingestion and Pipeline Setup

The first step in any ML pipeline is preparing the data. To make this reproducible (a core tenet of CI/CD), we wrap our data loading and preprocessing logic into clear, modular functions. We will use the classic Breast Cancer dataset for a binary classification task.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load_and_prep_data():
    """Loads dataset and performs standard train/test splits."""
    # 1. Load data
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    
    # 2. Split data (vital for evaluating model generalizing in CI)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 3. Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# Execute the pipeline step
X_train, X_test, y_train, y_test, scaler = load_and_prep_data()
print(f"Training data shape: {X_train.shape}")

## Step 2: Model Training and Experiment Tracking (The Core of ML CI)

If you train a model on your laptop, get great results, and forget the exact parameters you used, your pipeline is broken. 

We use **MLflow** to track everything. Every time this cell runs (e.g., when new code is pushed to a repository), MLflow logs:
* The hyperparameters used.
* The performance metrics (Accuracy, F1 Score).
* The actual model artifact itself.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

def train_and_log_model(X_train, X_test, y_train, y_test, n_estimators=100, max_depth=5):
    """Trains a Random Forest and logs all details to MLflow."""
    
    # Start an MLflow run to encapsulate this training iteration
    with mlflow.start_run(run_name="RandomForest_Pipeline_Run"):
        
        # 1. Initialize and train the model
        model = RandomForestClassifier(
            n_estimators=n_estimators, 
            max_depth=max_depth, 
            random_state=42
        )
        model.fit(X_train, y_train)
        
        # 2. Make predictions
        predictions = model.predict(X_test)
        
        # 3. Calculate metrics
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions)
        recall = recall_score(y_test, predictions)
        
        # 4. LOGGING TO MLFLOW (This is the crucial CI tracking step)
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        
        # Save the model artifact inside MLflow
        mlflow.sklearn.log_model(model, "random_forest_model")
        
        print(f"Run completed. Model Logged.")
        print(f"Metrics -> Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
        
        return model

# Execute the training pipeline step
trained_model = train_and_log_model(X_train, X_test, y_train, y_test)

## Step 3: Continuous Integration (CI) - Automated Testing

In standard software, CI tests if your functions crash. In Machine Learning, a model can successfully compile and output predictions, but those predictions might be terrible. 

Therefore, ML CI tests must evaluate **Model Quality** and **Data Integrity**. Here, we use `pytest` conventions to write assertions. In a real repository, a GitHub Action would run these tests and block the deployment if they fail.

In [ ]:
# In a real repository, this would be a separate file named `test_pipeline.py`
# We are running it here inline to demonstrate the logic.

def test_data_quality(X_train):
    """Test to ensure no NaN values snuck into our training data."""
    import numpy as np
    assert not np.isnan(X_train).any(), "Data quality failure: Null values detected in training set!"
    print("✓ Data Quality Test Passed")

def test_model_accuracy(model, X_test, y_test):
    """Test to ensure the newly trained model meets baseline business requirements."""
    from sklearn.metrics import accuracy_score
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    
    # We require at least 90% accuracy to allow deployment
    MINIMUM_ACCURACY_THRESHOLD = 0.90
    
    assert accuracy >= MINIMUM_ACCURACY_THRESHOLD, f"Model Degradation! Accuracy {accuracy} is below {MINIMUM_ACCURACY_THRESHOLD}"
    print(f"✓ Model Performance Test Passed (Accuracy: {accuracy:.4f})")

# Execute our simulated CI pipeline tests
try:
    print("--- Running CI Tests ---")
    test_data_quality(X_train)
    test_model_accuracy(trained_model, X_test, y_test)
    print("--- CI Pipeline Status: SUCCESS ---")
except AssertionError as e:
    print(f"--- CI Pipeline Status: FAILED ---")
    print(f"Error: {e}")

## Step 4: Continuous Deployment (CD) - Model Serving via API

Once the model passes the CI tests, the final step is CD: putting the model into production. 
In modern ML architectures, this usually means wrapping the trained model in a REST API so other microservices, web apps, or mobile apps can send it data and receive predictions.

We use **FastAPI** because it is exceptionally fast, easy to write, and automatically generates documentation for our endpoint.

In [ ]:
# In a real repository, this would be `app.py` and run on a server.
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

# 1. Define the API schema (What data format does the API expect?)
# We use Pydantic to ensure incoming requests have the right structure.
class PredictionRequest(BaseModel):
    features: list[float]

# 2. Initialize the API
app = FastAPI(title="Breast Cancer Prediction API", version="1.0")

# 3. Create the deployment endpoint
@app.post("/predict")
def predict(request: PredictionRequest):
    """
    Accepts an array of 30 feature values, scales them, 
    and returns a prediction using our CI-approved model.
    """
    # Convert input to the correct 2D numpy array shape for scikit-learn
    input_data = np.array(request.features).reshape(1, -1)
    
    # In a real app, we would load the scaler and model from disk or MLflow here.
    # Since we are in a notebook, we are using the globally available 'scaler' and 'trained_model'.
    scaled_data = scaler.transform(input_data)
    prediction = trained_model.predict(scaled_data)
    
    # Map the numeric prediction back to human-readable labels
    label = "Malignant" if prediction[0] == 0 else "Benign"
    
    return {
        "model_version": "RandomForest_v1",
        "prediction_class": int(prediction[0]),
        "prediction_label": label
    }

# --- Demonstration Code ---
# Since we cannot easily run a persistent web server inside a single notebook cell 
# without blocking it, here is how you would test the logic of the FastAPI endpoint:

print("Simulating an API Request via the CD pipeline...")
sample_patient_data = X_test[0].tolist() # Taking one sample from our test set
mock_request = PredictionRequest(features=sample_patient_data)

api_response = predict(mock_request)
print("API Response:", api_response)

## Summary

You have just built the logical foundation of an ML CI/CD pipeline:
1.  **Modularized Data Loading:** Ensuring reproducibility.
2.  **Experiment Tracking (MLflow):** Logging parameters, metrics, and models so you never lose a successful run.
3.  **Automated Testing (pytest):** Hard-coding business and data rules to prevent bad models from reaching users.
4.  **Deployment (FastAPI):** Exposing the verified model as a scalable web service.

To take this to the next level in a production environment, you would place this code in a Git repository and use a `.yaml` file (like GitHub Actions or GitLab CI) to orchestrate these steps automatically whenever a developer types `git push`.